## Send API requests

This notebook takes the shapefile of fire we want to validate, prepares all the API requests, sends them off and creates a log file of all sent jobs.

In [1]:
import pandas as pd
import geopandas as gpd
import fiona
import numpy as np
import requests
from src import config
from src import APIrequests

In [2]:
calfire_validation = gpd.read_file(config.PATH_FIRES_VALIDATION)
calfire_validation.head()
fire_names = calfire_validation['FIRE_NAME'].unique()
results = []

### Test if it works

In [3]:
fire_name = 'YORK'
print(fire_name)
bbox = APIrequests.create_bbox(fire_name, calfire_validation)
date_of_fire = APIrequests.get_fire_date(fire_name, calfire_validation)
query_data = APIrequests.create_query(fire_name, bbox.values[0], date_of_fire, 15)
try:
    # Use the 'json' parameter: it automatically sets 'Content-Type: application/json'
    # and runs json.dumps() for you.
    response = requests.post(config.URL_API, json=query_data)

    # 4. Check the results
    if response.status_code == 200 or response.status_code == 201:
        print("Success!")
        print(response.json()) # This is the data the API sends back
    else:
        print(f"Failed with status code: {response.status_code}")
        print(response.text) # This shows the error message from the API

except requests.exceptions.RequestException as e:
    print(f"A connection error occurred: {e}")

YORK
getting fire date
Success!
{'fire_event_name': 'YORK_date2023-07-28_range15_modealarm_sensorsentinel-2', 'status': 'Processing started', 'job_id': 'eb96eb16-e04b-486e-9861-3152348e22ff'}


### Send off all jobs

In [5]:

# Loop through each fire, date mode, sensor, and post-fire day range
for fire_name in fire_names:
    bbox = APIrequests.create_bbox(fire_name, calfire_validation)

    for post_fire_reference_point in config.POST_FIRE_REFERENCE_POINT:
        if post_fire_reference_point == 'alarm':
            date_of_fire = APIrequests.get_fire_date(fire_name, calfire_validation)
        else:
            date_of_fire = APIrequests.get_cont_date(fire_name, calfire_validation)

        if date_of_fire is None:
            for sensor in config.SENSORS:
                for post_fire_period in config.POST_FIRE_PERIOD:
                    APIrequests.append_result(results, fire_name, post_fire_reference_point, post_fire_period, sensor, 'skipped_no_cont_date')
            print(f"- {fire_name} ({post_fire_reference_point}): skipped — no CONT_DATE")
            continue

        for sensor in config.SENSORS:
            print(f"\n=== {fire_name} ({post_fire_reference_point}) — sensor: {sensor} ===")
            for post_fire_period in config.POST_FIRE_PERIOD:
                query_data = APIrequests.create_query(fire_name, bbox.values[0], date_of_fire, post_fire_period, post_fire_reference_point, sensor)

                try:
                    response = requests.post(config.URL_API, json=query_data)

                    if response.status_code == 200 or response.status_code == 201:
                        response_data = response.json()
                        APIrequests.append_result(results, fire_name, post_fire_reference_point, post_fire_period, sensor, 'success',
                                   fire_event_name=response_data.get('fire_event_name'),
                                   job_id=response_data.get('job_id'))
                        print(f"✓ [{sensor}] {fire_name} ({post_fire_reference_point}, {post_fire_period} days): {response_data.get('job_id')}")
                    else:
                        APIrequests.append_result(results, fire_name, post_fire_reference_point, post_fire_period, sensor, f'failed_{response.status_code}',
                                    fire_event_name=query_data['fire_event_name'])
                        print(f"✗ [{sensor}] {fire_name} ({post_fire_reference_point}, {post_fire_period} days): Failed with {response.status_code}")

                except requests.exceptions.RequestException as e:
                    APIrequests.append_result(results, fire_name, post_fire_reference_point, post_fire_period, sensor, 'error',
                                    fire_event_name=query_data['fire_event_name'])
                    print(f"✗ [{sensor}] {fire_name} ({post_fire_reference_point}, {post_fire_period} days): Connection error")

# Create dataframe and save to CSV
df_results = pd.DataFrame(results)
df_results.to_csv(config.PATH_JOBS_LOG, index=False)

print(f"\nProcessed {len(results)} requests. Results saved to {config.PATH_JOBS_LOG}")
print(df_results['sensor'].value_counts())
display(df_results)


getting fire date

=== COFFEE POT (alarm) — sensor: sentinel-2 ===
✓ [sentinel-2] COFFEE POT (alarm, 5 days): c6930ec1-05a7-44a6-aeb4-69808b47925d
✓ [sentinel-2] COFFEE POT (alarm, 10 days): 7b80118b-30b5-4fff-b7af-33808db1eded
✓ [sentinel-2] COFFEE POT (alarm, 15 days): f922ed2f-d9b0-4266-a43a-352258672478
✓ [sentinel-2] COFFEE POT (alarm, 21 days): c9ec7a41-8303-4841-b83c-01e95c2d8b97
✓ [sentinel-2] COFFEE POT (alarm, 30 days): e55be861-b43e-496f-8e31-860f59bbff11
✓ [sentinel-2] COFFEE POT (alarm, 45 days): f7f3fce0-d087-465d-8b87-9a3f2df6428c
✓ [sentinel-2] COFFEE POT (alarm, 60 days): 8318f19d-e8cc-427f-a98b-abe74aa8c810
✓ [sentinel-2] COFFEE POT (alarm, 90 days): 84ae015b-05b1-4178-bac7-e5a298d8bbf0

=== COFFEE POT (alarm) — sensor: landsat ===
✓ [landsat] COFFEE POT (alarm, 5 days): 0c43da0b-dbc7-499c-aa76-caa305165439
✓ [landsat] COFFEE POT (alarm, 10 days): 6543ba35-4afe-4bdf-be32-5533561303ef
✓ [landsat] COFFEE POT (alarm, 15 days): 087725d0-0b9c-4df2-a68c-4b2980f7901f
✓ [land

,fire_event_name,job_id,fire_name,date_mode,post_fire_days,sensor,status
0,COFFEE POT_date2024-08-03_range5_modealarm_sen...,9917fe4d-81ad-4442-b4b7-fab0733b7d65,COFFEE POT,alarm,5,sentinel-2,success
1,COFFEE POT_date2024-08-03_range10_modealarm_se...,0792c7b9-6b79-4ea3-be07-534bad7a89b9,COFFEE POT,alarm,10,sentinel-2,success
2,COFFEE POT_date2024-08-03_range15_modealarm_se...,844ff2ba-bbf0-4fb7-aeb9-134c39107172,COFFEE POT,alarm,15,sentinel-2,success
3,COFFEE POT_date2024-08-03_range21_modealarm_se...,cc335bf7-db5b-4e2a-852a-dd926a7601ca,COFFEE POT,alarm,21,sentinel-2,success
4,COFFEE POT_date2024-08-03_range30_modealarm_se...,edcc8fee-ef25-4756-af94-bd06fd86a2f6,COFFEE POT,alarm,30,sentinel-2,success
...,...,...,...,...,...,...,...
1313,LIBERTY CANYON_date2016-11-05_range21_modecont...,a58a6333-1e49-4e10-be96-fdcd151356ed,LIBERTY CANYON,cont,21,landsat,success
1314,LIBERTY CANYON_date2016-11-05_range30_modecont...,4449b7e4-7aec-4c52-8d21-f0b691ca6a0e,LIBERTY CANYON,cont,30,landsat,success
1315,LIBERTY CANYON_date2016-11-05_range45_modecont...,4a31759f-6196-4806-994a-d3eb9050081c,LIBERTY CANYON,cont,45,landsat,success
1316,LIBERTY CANYON_date2016-11-05_range60_modecont...,98fb7fee-f3bf-42af-9cf2-42c839ea4c00,LIBERTY CANYON,cont,60,landsat,success


### Check status

In [6]:
fires = pd.read_csv(config.PATH_JOBS_LOG)

# Collect status for each row
statuses = []

for idx, row in fires.iterrows():
    if row['status'] != 'success':
        statuses.append(999)
        continue

    request = requests.get(f"{config.URL_RESULT}/{row['fire_event_name']}/{row['job_id']}")
    status = request.json().get('status')
    print(f"Job {row['fire_event_name']}: {status}")
    statuses.append(status)

# Add status column to dataframe
fires['job_status'] = statuses

# Save updated dataframe
fires.to_csv(config.PATH_JOBS_LOG, index=False)

print(f"\nUpdated {config.PATH_JOBS_LOG} with job_status column")
fires.head()


Job COFFEE POT_date2024-08-03_range5_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-03_range10_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-03_range15_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-03_range21_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-03_range30_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-03_range45_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-03_range5_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-03_range10_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-03_range15_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-03_range21_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-03_range30_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-03_range45_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-03_range60_modealarm_sensorsentinel-2: complete
Job COFFEE POT_date2024-08-

,fire_event_name,job_id,fire_name,date_mode,post_fire_days,sensor,status,job_status
0,COFFEE POT_date2024-08-03_range5_modealarm_sen...,9917fe4d-81ad-4442-b4b7-fab0733b7d65,COFFEE POT,alarm,5,sentinel-2,success,complete
1,COFFEE POT_date2024-08-03_range10_modealarm_se...,0792c7b9-6b79-4ea3-be07-534bad7a89b9,COFFEE POT,alarm,10,sentinel-2,success,complete
2,COFFEE POT_date2024-08-03_range15_modealarm_se...,844ff2ba-bbf0-4fb7-aeb9-134c39107172,COFFEE POT,alarm,15,sentinel-2,success,complete
3,COFFEE POT_date2024-08-03_range21_modealarm_se...,cc335bf7-db5b-4e2a-852a-dd926a7601ca,COFFEE POT,alarm,21,sentinel-2,success,complete
4,COFFEE POT_date2024-08-03_range30_modealarm_se...,edcc8fee-ef25-4756-af94-bd06fd86a2f6,COFFEE POT,alarm,30,sentinel-2,success,complete
